# Notebook 02 — Recommender Models

**Goal:** Train three journal recommenders (TF-IDF, SBERT, LDA) and combine them into a Hybrid using Reciprocal Rank Fusion. Save all model artifacts to `models/`.

> **Personal note:** Building three separate models might seem like overkill, but each captures a different aspect of the text. TF-IDF is fast and lexical; SBERT is semantic (it knows 'deep learning' ≈ 'neural network'); LDA is probabilistic (it finds latent topics). By fusing them with RRF we get the best of all three worlds. This multi-method approach also gives me a good story for the report's results section.


In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path("..").resolve()))

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

# Load the processed DataFrame from notebook 01
df = pd.read_parquet("../data/processed/processed_df.parquet")
print(f"Loaded: {len(df):,} articles, {df['journal_id'].nunique():,} journals")
df.head(2)


Loaded: 20,944 articles, 410 journals


,record_id,journal_id,title,pub_year,cite_count,impact_factor,journal_name,journal_issn,journal_abbr,abstract,keywords,keywords_plus,subjects,abstract_clean_len,processed_text
0,88652,11050,An updated survey of GA-based multiobjective o...,2000,337,None,ACM COMPUTING SURVEYS,0360-0300,ACM COMPUT SURV,<p>After using evolutionary techniques for sin...,"[algorithms, artificial intelligence, genetic ...","[GENETIC ALGORITHM, MULTICRITERIA OPTIMIZATION...","[Computer Science, Theory & Methods, Computer ...",153,evolutionary single objective optimization dec...
1,88653,11050,The state of the art in distributed query proc...,2000,298,None,ACM COMPUTING SURVEYS,0360-0300,ACM COMPUT SURV,<p>Distributed data processing is becoming a r...,"[query optimization, query execution, client-s...","[DATABASE-SYSTEMS, DATA REPLICATION, PERFORMAN...","[Computer Science, Theory & Methods, Computer ...",220,distribute processing reality business want re...


## Method A — TF-IDF + Cosine Similarity

### Why TF-IDF?

TF-IDF (Term Frequency–Inverse Document Frequency) is the go-to baseline for text retrieval. `sublinear_tf=True` applies log-scaling to term frequencies, which prevents very common terms from dominating. Bigrams (`ngram_range=(1,2)`) capture phrases like *machine learning*, *neural network*, *distributed system* that are highly discriminative between CS sub-fields.

Each journal is represented as the **mean TF-IDF vector** of all its articles. At query time, the input abstract is transformed into the same vector space and the top-5 journals are ranked by cosine similarity.


In [2]:
from src.recommender import TFIDFRecommender

tfidf_rec = TFIDFRecommender(
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.85,
    max_features=60_000,
    sublinear_tf=True,
)
tfidf_rec.fit(df, text_col="processed_text")
tfidf_rec.save()


[TF-IDF] Fitting vectorizer …


[TF-IDF] Computing per-journal centroids …


[TF-IDF] Ready — 410 journals, vocab=60,000


[TF-IDF] Saved to C:\Users\zehra\Masaüstü\dm\journal-finder\models\tfidf_recommender.pkl


### Quick sanity check — feed a known abstract back in

In [3]:
from src.preprocessing import process_abstract_only

sample_abstract = df["abstract"].iloc[0]
sample_journal  = df["journal_name"].iloc[0]
query = process_abstract_only(sample_abstract)

recs = tfidf_rec.recommend(query, top_k=5)
print(f"True journal: {sample_journal}")
print("\nTF-IDF Top-5 recommendations:")
for r in recs:
    match = "✓" if r["journal_name"] == sample_journal else " "
    print(f"  [{match}] #{r['rank']}  {r['journal_name']:<60}  score={r['score']:.4f}")

top_terms = tfidf_rec.get_top_terms(query, top_n=10)
print(f"\nTop query terms: {top_terms}")


True journal: ACM COMPUTING SURVEYS

TF-IDF Top-5 recommendations:
  [ ] #1  ADVANCES IN COMPUTERS, VOL 98                                 score=0.2304
  [ ] #2  EVOLUTIONARY COMPUTATION                                      score=0.1928
  [✓] #3  ACM COMPUTING SURVEYS                                         score=0.1766
  [ ] #4  IEEE TRANSACTIONS ON EVOLUTIONARY COMPUTATION                 score=0.1574
  [ ] #5  SWARM AND EVOLUTIONARY COMPUTATION                            score=0.1503

Top query terms: ['evolutionary', 'emphasize importance', 'objective fitness', 'way exploit', 'literature purpose', 'motivate researcher', 'variation exist', 'function finally', 'finally future', 'information current']


## Method B — SBERT Sentence Embeddings + Cosine Similarity

### Why SBERT?

`all-MiniLM-L6-v2` is a distilled Sentence-BERT model that encodes sentences into 384-dimensional dense vectors. The key advantage over TF-IDF: semantically equivalent phrases get *similar embeddings*, even without shared tokens. A query about "convolutional neural networks for image recognition" will find journals about "deep learning in computer vision" because SBERT understands the semantic relationship.

> **Note:** The first run downloads the model (~90 MB) and encodes all 23,801 documents — this takes 10–20 minutes on CPU. The embeddings are cached to `data/processed/sbert_embeddings.npy` so subsequent runs are instant.


In [4]:
from src.recommender import SBERTRecommender

sbert_rec = SBERTRecommender()
sbert_rec.fit(df, text_col="processed_text", batch_size=64, cache_embeddings=True)
sbert_rec.save()


[SBERT] Loading model 'all-MiniLM-L6-v2' …



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2454.26it/s]


BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[SBERT] Loading cached embeddings from C:\Users\zehra\Masaüstü\dm\journal-finder\data\processed\sbert_embeddings.npy


[SBERT] Ready — 410 journals, embed_dim=384
[SBERT] Saved to C:\Users\zehra\Masaüstü\dm\journal-finder\models\sbert_recommender.pkl


In [5]:
recs = sbert_rec.recommend(query, top_k=5)
print(f"True journal: {sample_journal}")
print("\nSBERT Top-5 recommendations:")
for r in recs:
    match = "✓" if r["journal_name"] == sample_journal else " "
    print(f"  [{match}] #{r['rank']}  {r['journal_name']:<60}  score={r['score']:.4f}")


True journal: ACM COMPUTING SURVEYS

SBERT Top-5 recommendations:
  [ ] #1  EVOLUTIONARY COMPUTATION                                      score=0.7435
  [ ] #2  IEEE TRANSACTIONS ON EVOLUTIONARY COMPUTATION                 score=0.7430
  [ ] #3  SWARM AND EVOLUTIONARY COMPUTATION                            score=0.7028
  [ ] #4  MEMETIC COMPUTING                                             score=0.6971
  [ ] #5  INTERNATIONAL JOURNAL OF BIO-INSPIRED COMPUTATION             score=0.6882


## Method C — LDA Topic Model + Jensen–Shannon Divergence

### Why LDA?

Latent Dirichlet Allocation models documents as mixtures of topics. A paper about "security in distributed cloud systems" has high probability on both a *security* topic and a *distributed systems* topic. Jensen–Shannon divergence measures how different two probability distributions are — so we find the journal whose *topic mixture* is closest to the query's topic mixture.

I chose **K=50 topics** based on the coherence sweep in Notebook 04. The LDA also feeds directly into the topic clustering analysis.

> **Note:** LDA training takes ~5–10 minutes on the full corpus.


In [6]:
from src.recommender import LDARecommender

lda_rec = LDARecommender(n_topics=50, random_state=42, passes=10, workers=1)
lda_rec.fit(df, text_col="processed_text")
lda_rec.save()


[LDA] Building dictionary …


[LDA] Training LDA (K=50, passes=10) …


[LDA] Computing document topic distributions …


[LDA] Ready — 410 journals, 50 topics.


[LDA] Saved to C:\Users\zehra\Masaüstü\dm\journal-finder\models\lda_recommender.pkl


In [7]:
recs = lda_rec.recommend(query, top_k=5)
print(f"True journal: {sample_journal}")
print("\nLDA Top-5 recommendations:")
for r in recs:
    match = "✓" if r["journal_name"] == sample_journal else " "
    print(f"  [{match}] #{r['rank']}  {r['journal_name']:<60}  score={r['score']:.4f}")

print("\nTop topics in this LDA model (Topic 0–9):")
for tid in range(10):
    words = lda_rec.get_topic_words(tid, top_n=8)
    print(f"  Topic {tid:2d}: {', '.join(words)}")


True journal: ACM COMPUTING SURVEYS

LDA Top-5 recommendations:
  [ ] #1  ADVANCES IN COMPUTERS, VOL 98                                 score=0.5000
  [ ] #2  SWARM AND EVOLUTIONARY COMPUTATION                            score=0.4787
  [ ] #3  INFORMATION SYSTEMS MANAGEMENT                                score=0.4545
  [ ] #4  IEEE TRANSACTIONS ON EVOLUTIONARY COMPUTATION                 score=0.4469
  [ ] #5  COMPUTERS & OPERATIONS RESEARCH                               score=0.4414

Top topics in this LDA model (Topic 0–9):
  Topic  0: control, distance, similarity, measure, attribute, controller, base, xml
  Topic  1: flow, surface, element, simulation, structure, finite, numerical, mesh
  Topic  2: network, node, protocol, hoc, rout, telecommunication, wireless, traffic
  Topic  3: service, search, web, location, quality, qos, composition, base
  Topic  4: logic, language, program, semantic, verification, theory, transformation, formal
  Topic  5: semantic, knowledge, ontology, inf

## Method D — Hybrid (Reciprocal Rank Fusion)

### Why RRF?

Reciprocal Rank Fusion (Cormack et al., 2009) combines rankings without assuming scores are on comparable scales. The RRF score for journal $j$ is:

$$\text{RRF}(j) = \sum_{m} \frac{1}{k + \text{rank}_m(j)}$$

where $k=60$ is a smoothing constant. RRF is simple, parameter-free (besides $k$), and consistently outperforms score-averaging in information retrieval benchmarks.


In [8]:
from src.recommender import HybridRecommender

hybrid = HybridRecommender(tfidf=tfidf_rec, sbert=sbert_rec, lda=lda_rec)
recs = hybrid.recommend(query, top_k=5)

print(f"True journal: {sample_journal}")
print("\nHYBRID (RRF) Top-5 recommendations:")
for r in recs:
    match = "✓" if r["journal_name"] == sample_journal else " "
    tfidf_s = r.get("score_tfidf", float("nan"))
    sbert_s = r.get("score_sbert", float("nan"))
    lda_s   = r.get("score_lda",   float("nan"))
    print(f"  [{match}] #{r['rank']}  {r['journal_name']:<55}  "
          f"RRF={r['score']:.5f}  TF={tfidf_s:.3f}  SB={sbert_s:.3f}  LDA={lda_s:.3f}")


True journal: ACM COMPUTING SURVEYS

HYBRID (RRF) Top-5 recommendations:
  [ ] #1  SWARM AND EVOLUTIONARY COMPUTATION                       RRF=0.04739  TF=0.150  SB=0.703  LDA=0.479
  [ ] #2  IEEE TRANSACTIONS ON EVOLUTIONARY COMPUTATION            RRF=0.04738  TF=0.157  SB=0.743  LDA=0.447
  [ ] #3  EVOLUTIONARY COMPUTATION                                 RRF=0.04702  TF=0.193  SB=0.743  LDA=0.436
  [ ] #4  ADVANCES IN COMPUTERS, VOL 98                            RRF=0.04687  TF=0.230  SB=0.627  LDA=0.500
  [ ] #5  MEMETIC COMPUTING                                        RRF=0.04593  TF=0.145  SB=0.697  LDA=0.439


In [9]:
# Save the hybrid (saves all three sub-models again, harmless)
hybrid.save()
print("All models saved to ../models/")


[TF-IDF] Saved to C:\Users\zehra\Masaüstü\dm\journal-finder\models\tfidf_recommender.pkl
[SBERT] Saved to C:\Users\zehra\Masaüstü\dm\journal-finder\models\sbert_recommender.pkl


[LDA] Saved to C:\Users\zehra\Masaüstü\dm\journal-finder\models\lda_recommender.pkl
[Hybrid] All components saved.
All models saved to ../models/


## Interactive Demo (in-notebook)

Enter your own abstract below to try the recommender!


In [10]:
my_abstract = """
We propose a transformer-based architecture for automated code generation
from natural language specifications. Our model combines pre-trained language
models with execution feedback to iteratively refine generated programs.
We evaluate on competitive programming benchmarks and show significant
improvements over prior baselines.
"""

query_clean = process_abstract_only(my_abstract)
print("=== HYBRID JOURNAL RECOMMENDATIONS ===")
for r in hybrid.recommend(query_clean, top_k=5):
    print(f"  #{r['rank']}  {r['journal_name']:<60}  RRF={r['score']:.5f}")


=== HYBRID JOURNAL RECOMMENDATIONS ===


  #1  ACM TRANSACTIONS ON PROGRAMMING LANGUAGES AND SYSTEMS         RRF=0.04865
  #2  INTERNATIONAL JOURNAL OF PARALLEL PROGRAMMING                 RRF=0.04717
  #3  COMPUTER LANGUAGES SYSTEMS & STRUCTURES                       RRF=0.04641
  #4  JOURNAL OF FUNCTIONAL PROGRAMMING                             RRF=0.04554
  #5  ACM TRANSACTIONS ON ARCHITECTURE AND CODE OPTIMIZATION        RRF=0.04465
